In [0]:
storage_account_name = "capstonestorageaccadls"  # Replace with your storage account name
secret_scope = "keyvaultcapstone"
secret_key = "access-key"

# Retrieve the storage account key from the secret scope
storage_account_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)

# Configure Spark to access ADLS Gen2 using the access key
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
%sql
use catalog mavenmarket;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
%sql
use bronze;

In [0]:
from pyspark.sql.functions import current_timestamp, to_date, col

# ---------------------------------------------------------
# 1. Configuration & Key Vault Secrets
# ---------------------------------------------------------
catalog_name = "mavenmarket"
schema_name = "bronze"
table_name = "transactions"

# 3-level Unity Catalog namespace
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

# Fetch the datalake path from the Key Vault backed Secret Scope
secret_scope = "capstone-key-vault-new"
source_path = dbutils.secrets.get(scope=secret_scope, key="datalake-path").strip() + "transactions/"

# Define checkpoint and schema paths using the Unity Catalog Volume
# Ensure there is a trailing slash before the table name directory
checkpoint_path = f"/Volumes/{catalog_name}/{schema_name}/checkpoints/{table_name}"
schema_path = f"/Volumes/{catalog_name}/{schema_name}/checkpoints/{table_name}_schema"

# ---------------------------------------------------------
# 2. Read Stream (Auto Loader with Rescued Data)
# ---------------------------------------------------------
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    # Rescued data configuration:
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

# ---------------------------------------------------------
# 3. Add Metadata Columns
# ---------------------------------------------------------
df_enriched = (
    df_stream
    .withColumn("ingested_timestamp", current_timestamp())
    .withColumn("batch_date", to_date(current_timestamp()))
    .withColumn("file_source", col("_metadata.file_path"))
)

# ---------------------------------------------------------
# 4. Write Stream to Delta Table (Append Only)
# ---------------------------------------------------------
query = (
    df_enriched.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True) # Runs all available files, then stops
    # Defaults to outputMode("append"), ideal for downstream DLT CDC
    .toTable(full_table_name)
)

query.awaitTermination()

In [0]:
%sql
select * from mavenmarket.bronze.transactions